In [1]:
%pip install -U 'chronos-forecasting>=2.0' pandas

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from chronos import BaseChronosPipeline, Chronos2Pipeline
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')


from google.colab import drive
drive.mount('/content/drive')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.6/80.6 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 98.0 MB/s eta 0:00:00
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninstalled pandas-2.2.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.5 which is incompatible.
dask-cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.5 which is incompatible.
cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.5 which is incompatible.
Mounted at /content/drive


In [2]:
# Use only 1 GPU if available
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# Load the Chronos-2 pipeline
# GPU recommended for faster inference, but CPU is also supported using device_map="cpu"
pipeline: Chronos2Pipeline = BaseChronosPipeline.from_pretrained("amazon/chronos-2", device_map="cuda")

config.json:   0%|          | 0.00/1.07k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  478MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

## Model Set-up




*   Chronos (frozen) -> embeddings --> pooling
*   trainable head -> predictions

### Define Rolling Cross-Validation

In [3]:
def rolling_outer_split(
    n_samples,
    train_size,
    test_size,
    step_size
):
    """
    Yields (train_idx, test_idx)
    """
    splits = []
    start = 0

    while start + train_size + test_size <= n_samples:
        train_idx = range(start, start + train_size)
        test_idx = range(start + train_size,
                         start + train_size + test_size)
        splits.append((list(train_idx), list(test_idx)))
        start += step_size

    return splits

def rolling_inner_split(
    train_indices,
    val_size,
    step_size
):
    """
    Nested split inside outer train window
    """
    splits = []
    start = train_indices[0]
    end = train_indices[-1] + 1

    while start + val_size <= end:
        train_idx = list(range(train_indices[0], start))
        val_idx = list(range(start, start + val_size))

        if len(train_idx) > 0:
            splits.append((train_idx, val_idx))

        start += step_size

    return splits


### Model

In [4]:
import warnings

import torch
from torch.utils.data import DataLoader

from chronos.chronos2.dataset import Chronos2Dataset, DatasetMode


@torch.no_grad()
def embed_reg_tokens(
    pipeline,
    inputs,
    batch_size: int = 256,
    context_length: int | None = None,
):
    if context_length is None:
        context_length = pipeline.model_context_length

    if context_length > pipeline.model_context_length:
        warnings.warn(
            f"context_length {context_length} > model default "
            f"{pipeline.model_context_length}; resetting."
        )
        context_length = pipeline.model_context_length

    test_dataset = Chronos2Dataset(
        inputs,
        context_length=context_length,
        prediction_length=0,
        batch_size=batch_size,
        output_patch_size=pipeline.model_output_patch_size,
        mode=DatasetMode.TEST,
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=None,
        num_workers=0,
        pin_memory=pipeline.model.device.type == "cuda",
        shuffle=False,
        drop_last=False,
    )

    all_reg_embeds = []
    all_loc_scales = []

    for batch in test_loader:
        assert batch["future_target"] is None

        batch_context = batch["context"]
        batch_group_ids = batch["group_ids"]
        batch_target_idx_ranges = batch["target_idx_ranges"]

        encoder_outputs, (locs, scales), _, num_context_patches = (
            pipeline.model.encode(
                context=batch_context.to(
                    device=pipeline.model.device,
                    dtype=torch.float32,
                ),
                group_ids=batch_group_ids.to(pipeline.model.device),
                num_output_patches=0,
            )
        )

        hidden = encoder_outputs.last_hidden_state
        # [batch * num_variates, context_patches + 1, 768]

        reg_hidden = hidden[:, num_context_patches, :]
        # [batch * num_variates, 768]

        batch_reg_embeds = [
            reg_hidden[start:end].cpu()
            for start, end in batch_target_idx_ranges
        ]
        # List of tensors shaped [num_variates, 768]

        batch_loc_scales = list(
            zip(
                [
                    locs[start:end].cpu()
                    for start, end in batch_target_idx_ranges
                ],
                [
                    scales[start:end].cpu()
                    for start, end in batch_target_idx_ranges
                ],
            )
        )

        all_reg_embeds.extend(batch_reg_embeds)
        all_loc_scales.extend(batch_loc_scales)

    return all_reg_embeds, all_loc_scales

In [5]:
from torch import nn
from chronos.chronos2.layers import ResidualBlock

class RegTokenQuantileHead(nn.Module):
    """
    Chronos-2-style residual output head using the REG token
    from each variate.

    Input:
        reg_embeddings: [batch, num_variates, d_model]

    Output:
        quantile_predictions: [batch, 3]
    """

    def __init__(
        self,
        num_variates: int,
        d_model: int,
        h_dim: int,
        act_fn_name: str,
        dropout_p: float = 0.0,
    ) -> None:
        super().__init__()

        self.num_variates = num_variates
        self.d_model = d_model
        self.in_dim = num_variates * d_model #(NO AVERAGE)

        self.output_head = ResidualBlock(
            in_dim=self.in_dim,
            h_dim=h_dim,
            out_dim=3,  # q10, q50, q90
            act_fn_name=act_fn_name,
            dropout_p=dropout_p,
            use_layer_norm=False,
        )

    def forward(
        self,
        reg_embeddings: torch.Tensor,
    ) -> torch.Tensor:
        if reg_embeddings.ndim != 3:
            raise ValueError(
                "Expected REG embeddings with shape "
                "[batch, num_variates, d_model], "
                f"but received {tuple(reg_embeddings.shape)}."
            )

        batch_size, num_variates, d_model = reg_embeddings.shape

        if num_variates != self.num_variates:
            raise ValueError(
                f"Expected {self.num_variates} variates, "
                f"but received {num_variates}."
            )

        if d_model != self.d_model:
            raise ValueError(
                f"Expected embedding dimension {self.d_model}, "
                f"but received {d_model}."
            )

        features = reg_embeddings.flatten(start_dim=1) # NO AVERAGE

        # [batch, num_variates * d_model]

        quantile_predictions = self.output_head(features)
        # [batch, 3]

        return quantile_predictions

### Helper Functions: Train Epoch and Eval Epoch





In [6]:
def _unpack_batch(batch, device):
    combined = batch["combined"].to(device).float()
    # [B, num_variates, 768]

    y = batch["y"].to(device).float().view(-1)
    # [B]

    return combined, y


def train_epoch(
    model,
    loader,
    optimizer,
    criterion,
    device,
    plot=False,
):
    model.train()

    train_loss = 0.0
    all_median_preds = []
    all_targets = []

    for batch in loader:
        combined, y = _unpack_batch(batch, device)

        optimizer.zero_grad(set_to_none=True)

        preds = model(combined)
        # [B, 3]
        # columns: q10, q50, q90

        loss = criterion(preds, y)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()

        # Use q50 as the point prediction for plotting.
        median_preds = preds[:, 1]
        # [B]

        all_median_preds.append(
            median_preds.detach().cpu().numpy()
        )
        all_targets.append(
            y.detach().cpu().numpy()
        )

    if plot:
        all_median_preds = np.concatenate(
            all_median_preds
        ).ravel()

        all_targets = np.concatenate(
            all_targets
        ).ravel()

        plt.figure(figsize=(10, 3.5))
        plt.plot(
            all_targets,
            label="True values",
            marker=".",
        )
        plt.plot(
            all_median_preds,
            label="Predicted median",
            marker=".",
        )
        plt.xlabel("Sample index")
        plt.ylabel("Value")
        plt.title("Training: Predicted median vs true values")
        plt.legend()
        plt.ylim(0, 1)
        plt.tight_layout()
        plt.show()

    return train_loss / max(1, len(loader))

In [7]:
@torch.no_grad()
def eval_epoch(model, loader, criterion, device, plot=False):
    model.eval()
    total_loss = 0.0
    all_preds = []
    all_targets = []

    for batch in loader:
        combined, y = _unpack_batch(batch, device)

        preds = model(combined)
        y = y.view(-1)

        loss = criterion(preds, y)
        total_loss += loss.item()

        all_preds.append(
            preds.detach().cpu().numpy()
        )
        all_targets.append(
            y.detach().cpu().numpy()
        )

    if plot:
        all_preds = np.concatenate(
            all_preds,
            axis=0,
        )
        all_targets = np.concatenate(
            all_targets,
        ).ravel()

        q10_preds = all_preds[:, 0]
        q50_preds = all_preds[:, 1]
        q90_preds = all_preds[:, 2]

        r = np.corrcoef(q50_preds, all_targets)[0, 1]
        r2 = r2_score(all_targets, q50_preds)

        print("*" * 80)
        print(f"Median prediction: r = {r:.4f}, R² = {r2:.4f}")

        pred_std = q50_preds.std()
        true_std = all_targets.std()

        variance_ratio = (
            pred_std / true_std
            if true_std > 0
            else np.nan
        )

        print("Variance ratio:", variance_ratio)

        if np.isfinite(variance_ratio) and variance_ratio < 0.30:
            print("⚠️ Variance collapse detected")

        crossing_rate = np.mean(
            (q10_preds > q50_preds)
            | (q50_preds > q90_preds)
        )

        print(f"Quantile crossing rate: {crossing_rate:.2%}")

        sample_idx = np.arange(len(all_targets))

        plt.figure(figsize=(10, 3.5))

        plt.fill_between(
            sample_idx,
            q10_preds,
            q90_preds,
            alpha=0.25,
            label="Predicted q10–q90 interval",
        )

        plt.plot(
            sample_idx,
            all_targets,
            label="True values",
            marker="o",
        )

        plt.plot(
            sample_idx,
            q50_preds,
            label="Predicted median",
            marker="x",
        )

        plt.xlabel("Sample index")
        plt.ylabel("Value")
        plt.title("Validation: Quantile predictions vs true values")
        plt.legend()
        plt.ylim(0, 1)
        plt.tight_layout()
        plt.show()

    return total_loss / max(1, len(loader))

### Define Loss Function

In [8]:
import torch
import torch.nn as nn


class ChronosQuantileLoss(nn.Module):
    """
    Quantile loss structured like the Chronos-2 training loss.

    Expected:
        pred:   [batch, num_quantiles]
        target: [batch]

    Quantile order:
        [0.1, 0.5, 0.9]
    """

    def __init__(
        self,
        quantiles=(0.1, 0.5, 0.9),
    ):
        super().__init__()

        self.register_buffer(
            "quantiles",
            torch.tensor(quantiles, dtype=torch.float32),
        )

    def forward(
        self,
        pred: torch.Tensor,
        target: torch.Tensor,
    ) -> torch.Tensor:
        pred = pred.float()
        target = target.float().view(-1)

        if pred.ndim != 2:
            raise ValueError(
                "Expected predictions with shape "
                f"[batch, quantiles], found {tuple(pred.shape)}."
            )

        if pred.shape[0] != target.shape[0]:
            raise ValueError(
                f"Batch mismatch: pred {tuple(pred.shape)}, "
                f"target {tuple(target.shape)}."
            )

        if pred.shape[1] != len(self.quantiles):
            raise ValueError(
                f"Expected {len(self.quantiles)} quantiles, "
                f"found {pred.shape[1]}."
            )

        # Match the Chronos-2 shape convention:
        # pred:      [B, Q, H]
        # target:    [B, 1, H]
        # quantiles: [1, Q, 1]
        #
        # Here H = 1 because you predict one downstream value.
        pred = pred.unsqueeze(-1)
        # [B, 3, 1]

        target = target[:, None, None]
        # [B, 1, 1]

        quantiles = self.quantiles.view(1, -1, 1)
        # [1, 3, 1]

        quantile_loss = 2 * torch.abs(
            (target - pred)
            * (
                (target <= pred).float()
                - quantiles
            )
        )
        # [B, 3, 1]

        # Same aggregation logic as Chronos-2:
        # mean over horizon, sum over quantiles, mean over batch
        loss = (
            quantile_loss
            .mean(dim=-1)
            .sum(dim=-1)
            .mean()
        )

        return loss

### Helper Functions

In [9]:
import torch
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error
import copy
import math
import time
from pathlib import Path
import random


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # Optional stricter reproducibility
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def augment_2d(X, y, technique, seed=0): # Based on Harinis Code
    rng = np.random.default_rng(seed)

    X = np.asarray(X, dtype=np.float32)
    y = np.asarray(y, dtype=np.float32).reshape(-1)

    assert len(X) == len(y), f"X/y length mismatch: {len(X)} vs {len(y)}"

    if technique == "none":
        return (
            np.empty((0, X.shape[1]), dtype=np.float32),
            np.empty((0,), dtype=np.float32)
        )

    if technique == "jitter":
        col_std = X.std(axis=0, keepdims=True) + 1e-6
        Xn = X + rng.normal(0.0, 0.05, size=X.shape) * col_std
        return Xn.astype(np.float32), y.copy().astype(np.float32)

    if technique == "mixup":
        n = len(X)
        i = rng.integers(0, n, size=n)
        j = rng.integers(0, n, size=n)
        lam = rng.beta(0.2, 0.2, size=n).astype(np.float32)

        Xn = lam[:, None] * X[i] + (1 - lam[:, None]) * X[j]
        yn = lam * y[i] + (1 - lam) * y[j]

        return Xn.astype(np.float32), yn.astype(np.float32)

    if technique == "knn_interp":
        n = len(X)
        k = min(3, n - 1)

        if k <= 0:
            return (
                np.empty((0, X.shape[1]), dtype=np.float32),
                np.empty((0,), dtype=np.float32)
            )

        y_diff = np.abs(y[:, None] - y[None, :])
        np.fill_diagonal(y_diff, np.inf)

        nn_idx = np.argsort(y_diff, axis=1)[:, :k]

        a = rng.integers(0, n, size=n)
        p = nn_idx[a, rng.integers(0, k, size=n)]
        lam = rng.uniform(0.5, 0.8, size=n).astype(np.float32)

        Xn = lam[:, None] * X[a] + (1 - lam[:, None]) * X[p]
        yn = lam * y[a] + (1 - lam) * y[p]

        return Xn.astype(np.float32), yn.astype(np.float32)

    raise ValueError(f"Unknown augmentation technique: {technique}")

import numpy as np
import torch


def augment_multistream_embeds(
    combined,
    y,
    technique="jitter",
    seed=0
):
    """
    Supports embeddings shaped either:
      (N, D)
      (N, T, D)

    Only augments training embeddings.

    Returns:
      combined_aug, y_aug
    """

    combined_cpu = combined.detach().cpu().float()
    y_np = np.asarray(y, dtype=np.float32).reshape(-1)

    n = len(y_np)

    assert combined_cpu.shape[0] == n, (
        f"combined length {combined_cpu.shape[0]} != y length {n}"
    )

    if technique == "none":
        return combined_cpu, torch.tensor(y_np, dtype=torch.float32)

    original_shape = combined_cpu.shape[1:]

    # Flatten embeddings to 2D for augmentation
    X = combined_cpu.reshape(combined_cpu.shape[0], -1).numpy().astype(np.float32)

    X_synth, y_synth = augment_2d(
        X,
        y_np,
        technique=technique,
        seed=seed
    )

    # If no synthetic samples were created, return original data
    if len(X_synth) == 0:
        return combined_cpu, torch.tensor(y_np, dtype=torch.float32)

    # Reshape synthetic samples back to original embedding shape
    synth_combined = torch.tensor(
        X_synth,
        dtype=torch.float32
    ).reshape(len(X_synth), *original_shape)

    combined_aug = torch.cat([combined_cpu, synth_combined], dim=0)

    y_aug = np.concatenate([y_np, y_synth], axis=0).astype(np.float32)
    y_aug = torch.tensor(y_aug, dtype=torch.float32)

    assert combined_aug.shape[0] == len(y_aug)

    return combined_aug, y_aug


def pad_ragged_batch_left(x_list, pad_value=np.nan):
    lengths = [x.shape[0] for x in x_list]
    max_len = max(lengths)
    feat_dim = x_list[0].shape[1]

    out = np.full((len(x_list), max_len, feat_dim), pad_value, dtype=np.float32)

    for i, x in enumerate(x_list):
        out[i, -x.shape[0]:, :] = x  # left pad, real data ends at right edge

    return out


def embed_stream(
    pipeline,
    x_list,
):
    x_pad = pad_ragged_batch_left(x_list)

    # [B, time, variates] -> [B, variates, time]
    x_t = torch.as_tensor(
        x_pad,
        dtype=torch.float32,
    ).permute(0, 2, 1)

    reg_embeddings, _ = embed_reg_tokens(
        pipeline,
        x_t,
    )
    # List of B tensors, each [V, 768]

    reg_embeddings = torch.stack(
        reg_embeddings,
        dim=0,
    )
    # [B, V, 768]

    print(
        "REG embedding shape:",
        reg_embeddings.shape,
    )

    return reg_embeddings


def load_or_compute_embed(
    cache_path,
    compute_fn,
    force_recompute=True,
):
    cache_path = Path(cache_path)

    if cache_path.exists() and not force_recompute:
        return torch.load(
            cache_path,
            map_location="cpu",
        )

    embed = compute_fn()

    cache_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    torch.save(
        embed.cpu(),
        cache_path,
    )

    return embed


class MultiStreamEmbedDataset(Dataset):
    def __init__(self, combined, y):
        self.combined = combined
        self.y = y

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return {
            "combined": self.combined[idx],
            "y": self.y[idx]
        }

def build_model_from_hparams(hparams, device):
    head_hparams = hparams["head"]

    return RegTokenQuantileHead(
        num_variates=head_hparams["num_variates"],
        d_model=head_hparams["d_model"],
        h_dim= head_hparams["h_dim"],
        act_fn_name=head_hparams["act_fn_name"],
        dropout_p=head_hparams["dropout_p"],
    ).to(device)

def average_state_dicts(state_dicts):
    averaged = {}
    for key in state_dicts[0]:
        tensors = [state[key].detach().cpu() for state in state_dicts]
        first_tensor = tensors[0]
        if torch.is_floating_point(first_tensor):
            avg_tensor = torch.stack([tensor.float() for tensor in tensors], dim=0).mean(dim=0)
            averaged[key] = avg_tensor.to(first_tensor.dtype)
        else:
            averaged[key] = first_tensor.clone()
    return averaged

def predict_with_state(
    state_dict,
    hparams,
    embeds,
    device,
):
    model = build_model_from_hparams(
        hparams,
        device,
    )

    model.load_state_dict(state_dict)
    model.eval()

    with torch.no_grad():
        quantile_preds = model(
            embeds["combined"].to(device).float()
        ).cpu().numpy()
        # [N, 3]

    return quantile_preds

def train_single_seed_model(
    train_loader,
    val_loader,
    hparams,
    device,
    seed,
    max_epochs=100,
    patience=15,
):
    seed_everything(seed)

    model = build_model_from_hparams(
        hparams,
        device,
    )

    criterion = ChronosQuantileLoss(
        quantiles=(0.1, 0.5, 0.9),
    ).to(device)

    optimizer = optim.AdamW(
        model.parameters(),
        lr=hparams["lr"],
        betas=(0.9, 0.95),
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.5,
        patience=5,
    )

    counter = 0
    best_inner_val = float("inf")
    best_inner_state = None
    best_inner_epoch = None

    for epoch in range(max_epochs):
        plot_now = (epoch + 1) % 5 == 0

        epoch_loss = train_epoch(
            model,
            train_loader,
            optimizer,
            criterion,
            device,
            plot=plot_now,
        )

        val_loss = eval_epoch(
            model,
            val_loader,
            criterion,
            device,
            plot=plot_now,
        )

        scheduler.step(val_loss)

        print("=" * 80)
        print(
            f"[Seed {seed}] Epoch [{epoch + 1}/{max_epochs}] "
            f"- Train Loss: {epoch_loss:.4f}, "
            f"Val Loss: {val_loss:.4f}"
        )
        print("=" * 80)

        if math.isfinite(val_loss) and val_loss < best_inner_val:
            best_inner_val = val_loss
            best_inner_state = copy.deepcopy(
                model.state_dict()
            )
            best_inner_epoch = epoch + 1
            counter = 0
        else:
            counter += 1

            if counter >= patience:
                break

    if best_inner_state is None:
        raise RuntimeError(
            f"No valid model state found for seed {seed}."
        )

    return {
        "seed": seed,
        "val_loss": best_inner_val,
        "state": best_inner_state,
        "best_epoch": best_inner_epoch,
    }



def finetune_for_retraining(
    model,
    train_loader,
    criterion,
    optimizer,
    device,
    retrain_seed,
    retrain_epochs,
):
    seed_everything(retrain_seed)

    for epoch in range(retrain_epochs):
        epoch_loss = train_epoch(
            model,
            train_loader,
            optimizer,
            criterion,
            device,
            plot=False,
        )

        print("=" * 80)
        print(
            f"[Retrain Seed {retrain_seed}] "
            f"Epoch [{epoch + 1}/{retrain_epochs}] "
            f"- Train Loss: {epoch_loss:.4f}"
        )
        print("=" * 80)

    return model

### Main Loop

In [10]:
# =============================================================================
# Helpers
# =============================================================================

def get_labels_for_outcome(labels_all, outcome):
    """
    labels_all shape: [n_samples, 3]
    Columns: negative, cognitive, positive
    """
    if outcome == "negative":
        return labels_all[:, 0]
    elif outcome == "cognitive":
        return labels_all[:, 1]
    elif outcome == "positive":
        return labels_all[:, 2]
    else:
        raise ValueError(f"Unknown outcome: {outcome}")


def evaluate_selected_seed_models(
    seed_runs,
    hparams,
    val_embeds,
    y_val,
    device,
    selection_mode,
    top_k_ensemble,
):
    """
    Evaluate either:
    - the single best seed
    - an ensemble of the top-k seeds

    Predictions have shape [N, 3]:
        column 0 = q10
        column 1 = q50
        column 2 = q90
    """

    seed_runs = sorted(
        seed_runs,
        key=lambda item: item["val_loss"],
    )

    if selection_mode == "single":
        selected_models = seed_runs[:1]
    elif selection_mode == "ensemble":
        selected_models = seed_runs[
            : min(top_k_ensemble, len(seed_runs))
        ]
    else:
        raise ValueError(
            f"Unknown selection_mode: {selection_mode}"
        )

    val_predictions = [
        predict_with_state(
            model_info["state"],
            hparams,
            val_embeds,
            device,
        )
        for model_info in selected_models
    ]
    # List of arrays shaped [N, 3]

    selected_val_pred = np.mean(
        np.stack(val_predictions, axis=0),
        axis=0,
    )
    # [N, 3]

    criterion = ChronosQuantileLoss(
        quantiles=(0.1, 0.5, 0.9),
    )

    selected_val_loss = criterion(
        torch.as_tensor(
            selected_val_pred,
            dtype=torch.float32,
        ),
        torch.as_tensor(
            np.asarray(y_val).reshape(-1),
            dtype=torch.float32,
        ),
    ).item()

    return (
        selected_models,
        selected_val_loss,
        selected_val_pred,
    )


def clone_selected_models(selected_models):
    """
    Deep-copy selected models so later training does not
    accidentally modify them.
    """
    return [
        {
            "seed": model_info["seed"],
            "val_loss": model_info["val_loss"],
            "state": copy.deepcopy(
                model_info["state"]
            ),
            "best_epoch": model_info["best_epoch"],
        }
        for model_info in selected_models
    ]

def retrain_model_on_outer_train(
    train_embed_combined,
    y_train,
    best_hparams,
    model_info,
    device,
    retrain_epoch_scale=0.7,
    retrain_lr=1e-4,
    batch_size=64,
    aug_technique="none",
):
    """
    Retrain a fresh model on the full outer training set.

    The outer test set is not used during retraining.
    """

    model_seed = model_info["seed"]

    retrain_epochs = max(
        1,
        int(round(model_info["best_epoch"] * retrain_epoch_scale))
    )

    final_hparams = copy.deepcopy(best_hparams)
    final_hparams["lr"] = retrain_lr

    final_train_embed_aug, y_train_aug = augment_multistream_embeds(
        train_embed_combined,
        y_train,
        technique=aug_technique,
        seed=model_seed
    )

    final_train_ds = MultiStreamEmbedDataset(
        final_train_embed_aug,
        y_train_aug
    )

    # Monitoring loader uses non-augmented full train data.
    # This is not a true validation set, but the outer test set remains clean.
    final_monitor_ds = MultiStreamEmbedDataset(
        train_embed_combined,
        y_train
    )

    final_train_loader = DataLoader(
        final_train_ds,
        batch_size=batch_size,
        shuffle=True,
        drop_last=False
    )

    final_monitor_loader = DataLoader(
        final_monitor_ds,
        batch_size=batch_size,
        shuffle=False,
        drop_last=False
    )

    print(
        f"Retraining seed {model_seed} for {retrain_epochs} epochs "
        f"with retrain_lr={retrain_lr}"
    )

    retrained_result = train_single_seed_model(
        train_loader=final_train_loader,
        val_loader=final_monitor_loader,
        hparams=final_hparams,
        device=device,
        seed=model_seed,
        max_epochs=retrain_epochs,
        patience=retrain_epochs + 1
    )

    final_model = build_model_from_hparams(final_hparams, device)
    final_model.load_state_dict(retrained_result["state"])
    final_model.eval()

    return final_model, retrained_result, final_hparams


  # =============================================================================
# Nested CV function: combined-only model
# =============================================================================

def run_nested_cv_for_person(
    features_combined,
    labels,
    participant_id,
    outcome,
    outer_cfg,
    inner_cfg,
    device,
    hyperparams_grid,
    model_dirs,
    result_dirs,
    embed_dir,
    final_variants,
    max_epochs=100,
    patience=15,
    num_seed_runs=10,
    top_k_ensemble=5,
    retrain_epoch_scale=0.7,
    retrain_lr=1e-4,
    seed_base=42,
    aug_technique="none",
):

    selection_modes = sorted(
        set(item["selection_mode"] for item in final_variants)
    )

    outer_splits = rolling_outer_split(
        n_samples=len(labels),
        **outer_cfg
    )

    print("\n" + "#" * 80)
    print(f"Participant: {participant_id}")
    print(f"Outcome: {outcome}")
    print(f"Selection modes: {selection_modes}")
    print(f"Final variants: {[item['variant_tag'] for item in final_variants]}")
    print(f"Top-k ensemble: {top_k_ensemble}")
    print("#" * 80)


    results_by_variant = {}

    for variant_info in final_variants:
        variant_tag = variant_info["variant_tag"]
        result_dir_now = result_dirs[variant_tag]

        result_path = (
            result_dir_now
            / f"results_{participant_id}_{outcome}.pth"
        )

        if result_path.exists():
            loaded_results = torch.load(
                result_path,
                map_location="cpu",
                weights_only=False,
            )

            if not isinstance(loaded_results, list):
                raise TypeError(
                    f"Expected a list in {result_path}, "
                    f"but found {type(loaded_results).__name__}"
                )

            results_by_variant[variant_tag] = loaded_results

            completed_folds = {
                result["outer_i"]
                for result in loaded_results
            }

            print(
                f"Loaded {len(loaded_results)} existing results "
                f"for {variant_tag}: folds {sorted(completed_folds)}"
            )

        else:
            results_by_variant[variant_tag] = []
            print(f"No existing results found for {variant_tag}.")


    for outer_i, (train_idx, test_idx) in enumerate(outer_splits):

        completed_for_all_variants = all(
            any(
                result["outer_i"] == outer_i
                for result in results_by_variant[variant_info["variant_tag"]]
            )
            for variant_info in final_variants
        )

        if completed_for_all_variants:
            print(f"Skipping completed outer fold {outer_i}")
            continue

        y_train = labels[train_idx]
        y_test = labels[test_idx]

        X_tr_combined = features_combined[train_idx]
        X_test_combined = features_combined[test_idx]

        print("\n" + "=" * 80)
        print(f"OUTER {outer_i}")
        print("test_idx:", test_idx)
        print("y_test:", np.asarray(y_test).reshape(-1))
        print("combined test len:", len(X_test_combined))
        print("=" * 80)

        inner_splits = rolling_inner_split(
            train_idx,
            **inner_cfg
        )

        best_by_selection = {}

        for selection_mode in selection_modes:
            best_by_selection[selection_mode] = {
                "best_val": float("inf"),
                "best_hparams": None,
                "best_seed_models": None,
                "best_y_val": None,
                "best_val_features": None,
                "best_inner_i": None,
            }

        # ---------------------------------------------------------------------
        # Inner CV: train seed runs once, evaluate both single and ensemble
        # ---------------------------------------------------------------------

        for inner_i, (inner_train_idx, val_idx) in enumerate(inner_splits):

            y_tr = labels[inner_train_idx]
            y_val = labels[val_idx]

            X_inner_tr_combined = features_combined[inner_train_idx]
            X_inner_val_combined = features_combined[val_idx]

            inner_emb_dir = (
                embed_dir
                / f"participant_{participant_id}"
                / "shared"
                / f"outer{outer_i}"
                / f"inner{inner_i}"
            )

            inner_emb_dir.mkdir(parents=True, exist_ok=True)

            with torch.no_grad():
                tr_embed_combined = load_or_compute_embed(
                    inner_emb_dir / "train_combined.pt",
                    lambda: embed_stream(pipeline, X_inner_tr_combined)
                )

                val_embed_combined = load_or_compute_embed(
                    inner_emb_dir / "val_combined.pt",
                    lambda: embed_stream(pipeline, X_inner_val_combined)
                )

            print(f"Inner {inner_i}:")
            print("  train embed:", tr_embed_combined.shape)
            print("  val embed:", val_embed_combined.shape)

            aug_seed = seed_base + outer_i * 1000 + inner_i * 100

            tr_embed_combined_aug, y_tr_aug = augment_multistream_embeds(
                tr_embed_combined,
                y_tr,
                technique=aug_technique,
                seed=aug_seed
            )

            train_ds = MultiStreamEmbedDataset(
                tr_embed_combined_aug,
                y_tr_aug
            )

            val_ds = MultiStreamEmbedDataset(
                val_embed_combined,
                y_val
            )

            train_loader = DataLoader(
                train_ds,
                batch_size=64,
                shuffle=True,
                drop_last=False
            )

            val_loader = DataLoader(
                val_ds,
                batch_size=21,
                shuffle=False,
                drop_last=False
            )

            val_embeds = {
                "combined": val_embed_combined
            }

            for h_idx, hparams in enumerate(hyperparams_grid, start=1):

                print("  Quantile head:", hparams.get("head", {}))
                print("  Learning rate:", hparams.get("lr", None))

                seed_runs = []

                seed_offset = seed_base + outer_i * 1000 + inner_i * 100 + h_idx * 10
                seed_candidates = [
                    seed_offset + seed_j
                    for seed_j in range(num_seed_runs)
                ]

                # Train all seed models once for this hyperparameter set
                for seed in seed_candidates:
                    print(f"Running seed {seed} for hyperparameter set {h_idx}")

                    seed_result = train_single_seed_model(
                        train_loader=train_loader,
                        val_loader=val_loader,
                        hparams=hparams,
                        device=device,
                        seed=seed,
                        max_epochs=max_epochs,
                        patience=patience
                    )

                    seed_runs.append(seed_result)

                # Evaluate both single and ensemble using the same seed runs
                for selection_mode in selection_modes:

                    selected_models, selected_val_loss, selected_val_pred = (
                        evaluate_selected_seed_models(
                            seed_runs=seed_runs,
                            hparams=hparams,
                            val_embeds=val_embeds,
                            y_val=y_val,
                            device=device,
                            selection_mode=selection_mode,
                            top_k_ensemble=top_k_ensemble,
                        )
                    )

                    print(
                        f"{selection_mode} selected seeds for set {h_idx}: "
                        f"{[model_info['seed'] for model_info in selected_models]}"
                    )
                    print(
                        f"{selection_mode} val loss for set {h_idx}: "
                        f"{selected_val_loss:.4f}"
                    )

                    if (
                        math.isfinite(selected_val_loss)
                        and selected_val_loss < best_by_selection[selection_mode]["best_val"]
                    ):
                        best_by_selection[selection_mode]["best_val"] = selected_val_loss
                        best_by_selection[selection_mode]["best_hparams"] = copy.deepcopy(hparams)
                        best_by_selection[selection_mode]["best_seed_models"] = clone_selected_models(
                            selected_models
                        )
                        best_by_selection[selection_mode]["best_y_val"] = np.asarray(y_val).copy()
                        best_by_selection[selection_mode]["best_val_features"] = {
                            "combined": X_inner_val_combined
                        }
                        best_by_selection[selection_mode]["best_inner_i"] = inner_i

        for selection_mode in selection_modes:
            if (
                best_by_selection[selection_mode]["best_hparams"] is None
                or not best_by_selection[selection_mode]["best_seed_models"]
            ):
                raise RuntimeError(
                    f"No model selected for participant {participant_id}, "
                    f"outcome {outcome}, outer split {outer_i}, "
                    f"selection mode {selection_mode}."
                )

        # ---------------------------------------------------------------------
        # Prepare outer train and outer test embeddings once
        # ---------------------------------------------------------------------

        final_emb_dir = (
            embed_dir
            / f"participant_{participant_id}"
            / "shared"
            / f"outer{outer_i}"
            / "final"
        )

        final_emb_dir.mkdir(parents=True, exist_ok=True)

        with torch.no_grad():

            train_embed_combined = load_or_compute_embed(
                final_emb_dir / "outer_train_combined.pt",
                lambda: embed_stream(pipeline, X_tr_combined)
            )

            test_embed_combined = load_or_compute_embed(
                final_emb_dir / "outer_test_combined.pt",
                lambda: embed_stream(pipeline, X_test_combined)
            )

        assert train_embed_combined.shape[0] == len(y_train)
        assert test_embed_combined.shape[0] == len(y_test)

        # ---------------------------------------------------------------------
        # Save all four final variants
        # ---------------------------------------------------------------------

        for variant_info in final_variants:

            variant_tag = variant_info["variant_tag"]
            selection_mode = variant_info["selection_mode"]
            retrain_final_now = variant_info["retrain_final"]

            already_completed = any(
                result["outer_i"] == outer_i
                for result in results_by_variant[variant_tag]
            )

            if already_completed:
                print(
                    f"Skipping outer fold {outer_i} for {variant_tag}: "
                    "result already exists."
                )
                continue

            best_info = best_by_selection[selection_mode]

            best_val = best_info["best_val"]
            best_hparams = best_info["best_hparams"]
            best_seed_models = best_info["best_seed_models"]
            best_y_val = best_info["best_y_val"]
            best_val_features = best_info["best_val_features"]
            best_inner_i = best_info["best_inner_i"]

            model_dir_now = model_dirs[variant_tag]
            result_dir_now = result_dirs[variant_tag]



            with torch.no_grad():
                val_embed_combined = load_or_compute_embed(
                    final_emb_dir / f"{selection_mode}_best_inner{best_inner_i}_val_combined.pt",
                    lambda: embed_stream(pipeline, best_val_features["combined"])
                )

            test_preds = []
            val_preds = []
            final_model_infos = []

            for model_rank, model_info in enumerate(best_seed_models, start=1):

                model_seed = model_info["seed"]

                if retrain_final_now:

                    final_model, retrained_result, final_hparams = retrain_model_on_outer_train(
                        train_embed_combined=train_embed_combined,
                        y_train=y_train,
                        best_hparams=best_hparams,
                        model_info=model_info,
                        device=device,
                        retrain_epoch_scale=retrain_epoch_scale,
                        retrain_lr=retrain_lr,
                        batch_size=64,
                        aug_technique=aug_technique
                    )

                    final_info = {
                        "seed": model_seed,
                        "mode": "retrained",
                        "selection_mode": selection_mode,
                        "original_best_epoch": model_info["best_epoch"],
                        "retrain_epochs": max(
                            1,
                            int(round(model_info["best_epoch"] * retrain_epoch_scale))
                        ),
                        "original_inner_val_loss": model_info["val_loss"],
                        "final_hparams": final_hparams,
                    }

                else:

                    final_hparams = copy.deepcopy(best_hparams)

                    final_model = build_model_from_hparams(
                        final_hparams,
                        device
                    )

                    final_model.load_state_dict(model_info["state"])
                    final_model.eval()

                    final_info = {
                        "seed": model_seed,
                        "mode": "selected_inner_model_no_retrain",
                        "selection_mode": selection_mode,
                        "best_epoch": model_info["best_epoch"],
                        "inner_val_loss": model_info["val_loss"],
                        "final_hparams": final_hparams,
                    }

                model_path = (
                    model_dir_now
                    / f"participant_{participant_id}"
                    / outcome
                    / f"outer{outer_i}"
                    / f"seed_{model_seed}.pth"
                )

                model_path.parent.mkdir(parents=True, exist_ok=True)
                torch.save(final_model, model_path)

                with torch.no_grad():

                    member_test_pred = final_model(
                        test_embed_combined.to(device).float()
                    ).cpu().numpy()
                    # [N_test, 3]

                    member_val_pred = final_model(
                        val_embed_combined.to(device).float()
                    ).cpu().numpy()
                    # [N_val, 3]

                test_preds.append(member_test_pred)
                val_preds.append(member_val_pred)

                final_info["model_path"] = str(model_path)
                final_model_infos.append(final_info)

            y_pred_quantiles = np.mean(
                np.stack(test_preds, axis=0),
                axis=0,
            )
            # [N_test, 3]

            y_pred_val_quantiles = np.mean(
                np.stack(val_preds, axis=0),
                axis=0,
            )

            y_pred = y_pred_quantiles[:, 1]
            # q50: [N_test]

            y_pred_val = y_pred_val_quantiles[:, 1]
            # q50: [N_val]

            y_true = np.asarray(y_test).reshape(-1)


            print("\n" + "*" * 80)
            print(f"Participant {participant_id}, outcome {outcome}, outer split {outer_i}")
            print(f"Variant: {variant_tag}")
            print(f"Selection mode: {selection_mode}")
            print(f"Retrain final: {retrain_final_now}")
            print(f"Best inner val loss: {best_val:.4f}")
            print(f"Best hparams: {best_hparams}")
            print(f"Selected seeds: {[model_info['seed'] for model_info in best_seed_models]}")
            print("*" * 80)

            y_train_mean_array = np.full_like(y_true, y_train.mean())

            metadata_path = (
                model_dir_now
                / f"participant_{participant_id}"
                / outcome
                / f"outer{outer_i}"
                / "metadata.pth"
            )

            metadata_path.parent.mkdir(parents=True, exist_ok=True)

            torch.save(
                {
                    "participant_id": participant_id,
                    "outcome": outcome,
                    "outer_split": outer_i,
                    "variant_tag": variant_tag,
                    "selection_mode": selection_mode,
                    "retrain_final": retrain_final_now,
                    "top_k_ensemble": top_k_ensemble,
                    "effective_top_k": 1 if selection_mode == "single" else top_k_ensemble,
                    "best_inner_i": best_inner_i,
                    "best_inner_val_loss": best_val,
                    "best_hparams": best_hparams,
                    "selected_seeds": [
                        model_info["seed"]
                        for model_info in best_seed_models
                    ],
                    "final_model_infos": final_model_infos,
                },
                metadata_path
            )

            results_by_variant[variant_tag].append(
                {
                    "participant_id": participant_id,
                    "outcome": outcome,
                    "variant_tag": variant_tag,
                    "selection_mode": selection_mode,

                    "outer_i": outer_i,
                    "train_idx": np.asarray(train_idx),
                    "test_idx": np.asarray(test_idx),

                    "y_test": y_true,

                    # Existing point-prediction interface:
                    "y_pred": y_pred,
                    "y_val": best_y_val,
                    "y_pred_val": y_pred_val,

                    # New quantile outputs:
                    "quantiles": np.asarray(
                        [0.1, 0.5, 0.9],
                        dtype=np.float32,
                    ),
                    "y_pred_quantiles": y_pred_quantiles,
                    "y_pred_q10": y_pred_quantiles[:, 0],
                    "y_pred_q50": y_pred_quantiles[:, 1],
                    "y_pred_q90": y_pred_quantiles[:, 2],
                    "y_pred_val_quantiles": y_pred_val_quantiles,

                    "y_train": y_train,
                    "y_train_mean": y_train_mean_array,
                    "y_train_last": np.full_like(
                        y_true,
                        y_train[-1],
                    ),


                    "best_hparams": best_hparams,
                    "best_inner_i": best_inner_i,
                    "best_inner_val_loss": best_val,

                    "selected_seeds": [
                        model_info["seed"]
                        for model_info in best_seed_models
                    ],
                    "selected_inner_val_losses": [
                        model_info["val_loss"]
                        for model_info in best_seed_models
                    ],

                    "retrain_final": retrain_final_now,
                    "top_k_ensemble": top_k_ensemble,
                    "effective_top_k": (
                        1
                        if selection_mode == "single"
                        else len(best_seed_models)
                    ),

                    "model_paths": [
                        info["model_path"]
                        for info in final_model_infos
                    ],
                    "metadata_path": str(metadata_path),
                }
            )

            result_path = (
                result_dir_now
                / f"results_{participant_id}_{outcome}.pth"
            )

            result_path.parent.mkdir(parents=True, exist_ok=True)
            torch.save(results_by_variant[variant_tag], result_path)

            print(f"Saved {variant_tag} running results to: {result_path}")

    return results_by_variant

In [11]:
import copy
import math
from pathlib import Path
from itertools import product

import numpy as np
import pandas as pd
import torch
import torch.optim as optim

from torch.utils.data import DataLoader
from sklearn.metrics import mean_squared_error, r2_score


# =============================================================================
# Global settings
# =============================================================================

BASE_DIR = Path(
    "/content/drive/MyDrive/Colab_Notebooks/Anna_JacobsonColabDrive/"
    "SensingIndividualized"
)

BASE_DATA_DIR = BASE_DIR / "deeplearning_data" / "pred"
MODEL_TYPE = "Model3_reg_quantile"
EMBED_DIR = BASE_DIR / "embeddings_pred" / MODEL_TYPE





AUG_TECHNIQUES = [
    "none",
 ]

# Always save all four final versions
FINAL_VARIANTS = [
    {
        "variant_tag": "no_retrain_single",
        "selection_mode": "single",
        "retrain_final": False,
    },
#     {
#         "variant_tag": "retrain_single",
#         "selection_mode": "single",
#         "retrain_final": True,
#     },
#     {
#         "variant_tag": "no_retrain_ensemble",
#         "selection_mode": "ensemble",
#         "retrain_final": False,
#     },
#     {
#         "variant_tag": "retrain_ensemble",
#         "selection_mode": "ensemble",
#         "retrain_final": True,
#     },
 ]

MODEL_DIRS = {
    item["variant_tag"]: BASE_DIR / f"final_models_{MODEL_TYPE}_{item['variant_tag']}"
    for item in FINAL_VARIANTS
}

RESULT_DIRS = {
    item["variant_tag"]: BASE_DIR / "results_true_pred" / f"{MODEL_TYPE}_{item['variant_tag']}"
    for item in FINAL_VARIANTS
}

for d in list(MODEL_DIRS.values()) + list(RESULT_DIRS.values()):
    d.mkdir(parents=True, exist_ok=True)

print("Saving these variants:")
for item in FINAL_VARIANTS:
    tag = item["variant_tag"]
    print(f"  {tag}")
    print(f"    MODEL_DIR : {MODEL_DIRS[tag]}")
    print(f"    RESULT_DIR: {RESULT_DIRS[tag]}")


# =============================================================================
# Hyperparameter setup
# =============================================================================

participant_ids = ["00092", "00011","00485", "00551", "00012", "00095","00179", "00368", "00424", "00477"]

outcomes = ["negative", "cognitive", "positive"]

outer_cfg = dict(
    train_size=135,
    test_size=3,
    step_size=3
)

inner_cfg = dict(
    val_size=21,
    step_size=114
)

lr_opts = [
    0.001,
    0.0004,
]

num_seed_runs = 3

# Used only for ensemble variants
top_k_ensemble = 3

retrain_epoch_scale = 0.7
retrain_lr = 1e-4

h_dim_opts = [
    64,
    128,
]

dropout_opts = [
    0.0,
    0.1,
]

def make_hyperparams_grid(
    num_variates,
    pipeline,
):
    """
    Construct hyperparameter combinations for the
    REG-token quantile prediction head.
    """

    return [
        {
            "head": {
                "num_variates": num_variates,
                "d_model": pipeline.model.config.d_model,
                "h_dim": h_dim,
                "act_fn_name": pipeline.model.config.dense_act_fn,
                "dropout_p": dropout_p,
            },
            "lr": lr,
        }
        for lr in lr_opts
        for h_dim in h_dim_opts
        for dropout_p in dropout_opts
    ]

print("Saving all variants:")
for item in FINAL_VARIANTS:
    print("  ", item["variant_tag"])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

all_results = {}
all_test_losses = {}


# =============================================================================
# Participant/outcome loop
# =============================================================================

for participant_id in participant_ids:

    file_path = BASE_DATA_DIR / f"participant_{participant_id}_4h.npz"

    print("\n" + "=" * 80)
    print(f"Participant: {participant_id}")
    print("=" * 80)

    try:
        loaded = np.load(file_path, allow_pickle=True)
    except FileNotFoundError:
        print(f"File not found, skipping participant {participant_id}: {file_path}")
        continue

    datetimes = pd.to_datetime(loaded["datetimes"])
    order = np.argsort(datetimes.values)

    labels_all = loaded["labels"].astype(np.float32)

    # Combined head only
    features_combined = loaded["combined"]

    # Sort by datetime
    features_combined = features_combined[order]
    labels_all = labels_all[order]
    datetimes = datetimes[order]

    # Scale / round labels
    labels_all = (np.round(labels_all)).astype(np.float32) / 100

    # Remove first questionnaire
    features_combined = features_combined[1:]
    labels_all = labels_all[1:]
    datetimes = datetimes[1:]

    first_sample = np.asarray(features_combined[0])

    if first_sample.ndim != 2:
        raise ValueError(
            "Expected each raw time-series example to have shape "
            f"[time, variates], found {first_sample.shape}."
        )

    num_variates = first_sample.shape[1]

    for sample_i, sample in enumerate(features_combined):
        sample = np.asarray(sample)

        if sample.ndim != 2:
            raise ValueError(
                f"Sample {sample_i} has shape {sample.shape}; "
                "expected [time, variates]."
            )

        if sample.shape[1] != num_variates:
            raise ValueError(
                f"Sample {sample_i} has {sample.shape[1]} variates, "
                f"but the first sample has {num_variates}."
            )

    hyperparams_grid = make_hyperparams_grid(
        num_variates=num_variates,
        pipeline=pipeline,
    )

    print("Number of variates:", num_variates)
    print("Chronos d_model:", pipeline.model.config.d_model)
    print("Chronos d_ff:", pipeline.model.config.d_ff)
    print("Quantile-head hyperparameter sets:", len(hyperparams_grid))

    for outcome in outcomes:

        print("\n" + "-" * 80)
        print(f"Running participant {participant_id}, outcome: {outcome}")
        print("-" * 80)

        labels = get_labels_for_outcome(labels_all, outcome)

        for aug_technique in AUG_TECHNIQUES:

            print("\n" + "=" * 60)
            print(f"Augmentation: {aug_technique}")
            print("=" * 60)

            # Separate folders for each augmentation.
            folder_aug = "" if aug_technique == "none" else f"_{aug_technique}"

            MODEL_DIRS_AUG = {
                item["variant_tag"]: (
                    BASE_DIR
                    / f"final_models_{MODEL_TYPE}{folder_aug}_{item['variant_tag']}"
                )
                for item in FINAL_VARIANTS
            }

            RESULT_DIRS_AUG = {
                item["variant_tag"]: (
                    BASE_DIR
                    / "results_true_pred"
                    / f"{MODEL_TYPE}{folder_aug}_{item['variant_tag']}"
                )
                for item in FINAL_VARIANTS
            }

            for d in list(MODEL_DIRS_AUG.values()) + list(RESULT_DIRS_AUG.values()):
                d.mkdir(parents=True, exist_ok=True)

            try:
                results_by_variant = run_nested_cv_for_person(
                    features_combined=features_combined,
                    labels=labels,
                    participant_id=participant_id,
                    outcome=outcome,
                    outer_cfg=outer_cfg,
                    inner_cfg=inner_cfg,
                    device=device,
                    hyperparams_grid=hyperparams_grid,
                    model_dirs=MODEL_DIRS_AUG,
                    result_dirs=RESULT_DIRS_AUG,
                    embed_dir=EMBED_DIR,
                    final_variants=FINAL_VARIANTS,
                    max_epochs=100,
                    patience=15,
                    num_seed_runs=num_seed_runs,
                    top_k_ensemble=top_k_ensemble,
                    retrain_epoch_scale=retrain_epoch_scale,
                    retrain_lr=retrain_lr,
                    seed_base=42,
                    aug_technique=aug_technique,
                )

                for variant_tag, results in results_by_variant.items():

                    test_losses = [
                        mean_squared_error(item["y_test"], item["y_pred"])
                        for item in results
                    ]

                    mean_test_loss = np.mean(test_losses)

                    result_key = (
                        participant_id,
                        outcome,
                        aug_technique,
                        variant_tag,
                    )

                    all_results[result_key] = results
                    all_test_losses[result_key] = {
                        "per_outer_split": test_losses,
                        "mean": mean_test_loss,
                    }

                    print(
                        f"\n{aug_technique} / {variant_tag} "
                        "per-outer-split test losses:"
                    )
                    print(test_losses)

                    print(
                        f"{aug_technique} / {variant_tag} "
                        f"mean test loss: {mean_test_loss}"
                    )

            except Exception as e:

                print(
                    f"Error for participant {participant_id}, "
                    f"outcome {outcome}, augmentation {aug_technique}: {e}"
                )

                for item in FINAL_VARIANTS:
                    variant_tag = item["variant_tag"]

                    result_key = (
                        participant_id,
                        outcome,
                        aug_technique,
                        variant_tag,
                    )

                    all_results[result_key] = None
                    all_test_losses[result_key] = None

Saving these variants:
  no_retrain_single
    MODEL_DIR : /content/drive/MyDrive/Colab_Notebooks/Anna_JacobsonColabDrive/SensingIndividualized/final_models_Model3_reg_quantile_no_retrain_single
    RESULT_DIR: /content/drive/MyDrive/Colab_Notebooks/Anna_JacobsonColabDrive/SensingIndividualized/results_true_pred/Model3_reg_quantile_no_retrain_single
Saving all variants:
   no_retrain_single

Participant: 00092
Number of variates: 8
Chronos d_model: 768
Chronos d_ff: 3072
Quantile-head hyperparameter sets: 8

--------------------------------------------------------------------------------
Running participant 00092, outcome: negative
--------------------------------------------------------------------------------

Augmentation: none

################################################################################
Participant: 00092
Outcome: negative
Selection modes: ['single']
Final variants: ['no_retrain_single']
Top-k ensemble: 3
########################################################